# Imports

In [63]:
from pathlib import Path
import h5py
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

## Globals

In [74]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_FOLDER = Path("data")
NYU_DATASET_FILE = DATA_FOLDER / "nyu_depth_v2_labeled.mat"
KITTI_DATASET_FOLDER = DATA_FOLDER / "kitti"

INPUT_SIZE = (224, 304)
BATCH_SIZE = 16 
EPOCH = 40
SEED = 42


## Utils

## Data

In [68]:
class NYUDataset(torch.utils.data.Dataset):
    def __init__(self,mat_file):
        with h5py.File(mat_file,"r") as f:
            self.images = f["images"][:]
            self.depths = f["depths"][:]

    def __len__(self):
        return len(self.images)

    def __getitem__(self,idx):
        img = self.images[idx]
        depth = self.depths[idx]
        img_t = np.transpose(img,(0,2,1))
        depth_t = depth.T
        image_tensor = torch.from_numpy(img_t).float() / 255
        depth_tensor = torch.from_numpy(depth_t).float()
        image_tensor = F.interpolate(
            image_tensor.unsqueeze(0),
            size = INPUT_SIZE,
            mode = "bilinear",
            align_corners = False
        ).squeeze(0)
        depth_tensor = F.interpolate(
            depth_tensor.unsqueeze(0).unsqueeze(0),
            size = INPUT_SIZE,
            mode = "nearest",
        ).squeeze(0)
        return image_tensor,depth_tensor


        

In [76]:
ds = NYUDataset(NYU_DATASET_FILE)
train_set_size = int(0.8 * len(ds))
val_set_size = len(ds) - train_set_size

#creates training and validation sets
training_set, validation_set = torch.utils.data.random_split(
    ds,
    [train_set_size,val_set_size],
    generator = torch.Generator().manual_seed(SEED)
)

#loads training and validation sets
train_loader = torch.utils.data.DataLoader(training_set , batch_size = BATCH_SIZE, shuffle = True)
val_loader = torch.utils.data.DataLoader(validation_set , batch_size = BATCH_SIZE, shuffle = True)

In [78]:
imgs,dep = next(iter(train_loader))
print(imgs.shape,dep.shape)

torch.Size([16, 3, 224, 304]) torch.Size([16, 1, 224, 304])


## Network

## Train

## Evaluation